# MobileNetV2 Hybrid Model — 5 Seed Experiment

This notebook keeps the same MobileNetV2 + custom Transformer + Capsule hybrid model structure and runs the experiment for 5 independent random seeds. It also reports model efficiency and ROC/AUC outputs for each seed, then summarizes mean ± std across seeds.

In [5]:
import os
import random
import time
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_v2_preprocess_input
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import AUC, Precision, Recall
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
# ============================================================
# 5-SEED EXPERIMENT SETTINGS
# ============================================================

import gc
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_curve, auc

SEEDS = [42, 123, 2024, 7, 99]

def set_all_seeds(seed):
    """Set Python, NumPy, and TensorFlow seeds for reproducible seed-wise experiments."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # This improves reproducibility where supported. GPU results can still have small nondeterminism.
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

print('Seeds used:', SEEDS)

In [3]:
import zipfile
import os

zip_path = "/content/archive (53).zip"
extract_path = "/content/extracted_data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted to:", extract_path)
print("Files:", os.listdir(extract_path))

Extracted to: /content/extracted_data
Files: ['dataset.csv', 'PMRAM Bangladeshi Brain Cancer - MRI Dataset']


In [4]:
TRAIN_PATH = "/content/extracted_data/PMRAM Bangladeshi Brain Cancer - MRI Dataset/PMRAM Bangladeshi Brain Cancer - MRI Dataset/Augmented Data/Augmented"
TEST_PATH = "/content/extracted_data/PMRAM Bangladeshi Brain Cancer - MRI Dataset/PMRAM Bangladeshi Brain Cancer - MRI Dataset/Raw Data/Raw"

In [7]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16

EMBED_DIM = 96
NUM_HEADS = 4
TRANSFORMER_DEPTH = 1
MLP_DIM = 192

PRIMARY_CAPS = 12
PRIMARY_CAPS_DIM = 8
CLASS_CAPS_DIM = 12
ROUTING_ITERS = 2
NUM_CLASSES = 4
ALPHA_CE = 0.9
BETA_MARGIN = 0.1
DROPOUT = 0.30
LABEL_SMOOTHING = 0.08
REG = tf.keras.regularizers.l2(1e-4)

In [ ]:
def build_generators(train_path, test_path, image_size=(224, 224), batch_size=16, validation_split=0.2, seed=42):
    train_datagen = ImageDataGenerator(
        preprocessing_function=mobilenet_v2_preprocess_input,
        rotation_range=10,
        width_shift_range=0.05,
        height_shift_range=0.05,
        zoom_range=0.06,
        brightness_range=(0.95, 1.05),
        fill_mode='nearest',
        validation_split=validation_split
    )

    val_datagen = ImageDataGenerator(
        preprocessing_function=mobilenet_v2_preprocess_input,
        validation_split=validation_split
    )

    test_datagen = ImageDataGenerator(
        preprocessing_function=mobilenet_v2_preprocess_input
    )

    train_gen = train_datagen.flow_from_directory(
        train_path,
        target_size=image_size,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=True,
        subset='training',
        seed=seed
    )

    val_gen = val_datagen.flow_from_directory(
        train_path,
        target_size=image_size,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False,
        subset='validation',
        seed=seed
    )

    test_gen = test_datagen.flow_from_directory(
        test_path,
        target_size=image_size,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False
    )
    return train_gen, val_gen, test_gen

In [9]:
from tensorflow.keras.applications import MobileNetV2


INPUT_SHAPE = (224, 224, 3)




def squash(vectors, axis=-1, epsilon=1e-7):
    s_squared_norm = tf.reduce_sum(tf.square(vectors), axis=axis, keepdims=True)
    scale = s_squared_norm / (1.0 + s_squared_norm)
    return scale * vectors / tf.sqrt(s_squared_norm + epsilon)


class PatchTokenization(layers.Layer):
    def __init__(self, patch_size=2, embed_dim=EMBED_DIM, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.proj = layers.Dense(embed_dim)

    def build(self, input_shape):
        _, h, w, c = input_shape
        self.num_patches_h = h // self.patch_size
        self.num_patches_w = w // self.patch_size
        self.num_tokens = self.num_patches_h * self.num_patches_w

        self.pos_embed = self.add_weight(
            name='pos_embed',
            shape=(1, self.num_tokens, self.embed_dim),
            initializer='random_normal',
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        patches = tf.image.extract_patches(
            images=x,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )
        batch_size = tf.shape(patches)[0]
        patch_dim = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, self.num_tokens, patch_dim])
        tokens = self.proj(patches)
        return tokens + self.pos_embed

In [10]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=DROPOUT, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=DROPOUT
        )
        self.drop1 = layers.Dropout(dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = tf.keras.Sequential([
            layers.Dense(mlp_dim, activation='gelu', kernel_regularizer=REG),
            layers.Dropout(dropout),
            layers.Dense(embed_dim,kernel_regularizer=REG),
            layers.Dropout(dropout)
        ])

    def call(self, x, training=False, return_attention=False):
        x_norm = self.norm1(x)
        attn_out, attn_scores = self.attn(
            x_norm, x_norm,
            return_attention_scores=True,
            training=training
        )
        x = x + self.drop1(attn_out, training=training)
        x = x + self.mlp(self.norm2(x), training=training)
        if return_attention:
            return x, attn_scores
        return x


class PrimaryCapsule(layers.Layer):
    def __init__(self, num_capsules=16, capsule_dim=8, **kwargs):
        super().__init__(**kwargs)
        self.num_capsules = num_capsules
        self.capsule_dim = capsule_dim
        self.proj = layers.Dense(num_capsules * capsule_dim,kernel_regularizer=REG)

    def call(self, tokens):
        x = self.proj(tokens)
        batch_size = tf.shape(x)[0]
        num_tokens = tf.shape(x)[1]
        x = tf.reshape(x, [batch_size, num_tokens, self.num_capsules, self.capsule_dim])
        return squash(x)

In [11]:
class RelevanceAwareClassCapsule(layers.Layer):
    def __init__(self, num_classes, class_caps_dim=CLASS_CAPS_DIM, routing_iters=ROUTING_ITERS, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.class_caps_dim = class_caps_dim
        self.routing_iters = routing_iters

    def build(self, input_shape):
        _, num_tokens, num_primary, primary_dim = input_shape[0]
        self.W = self.add_weight(
            shape=(1, num_tokens, num_primary, self.num_classes, self.class_caps_dim, primary_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='capsule_transform'
        )
        super().build(input_shape)

    def call(self, inputs):
        primary_caps, relevance_scores = inputs

        primary_caps_exp = tf.expand_dims(tf.expand_dims(primary_caps, axis=3), axis=-1)
        W_tiled = tf.tile(self.W, [tf.shape(primary_caps)[0], 1, 1, 1, 1, 1])
        u_hat = tf.matmul(W_tiled, primary_caps_exp)
        u_hat = tf.squeeze(u_hat, axis=-1)

        b = tf.zeros(
            shape=(tf.shape(primary_caps)[0], tf.shape(primary_caps)[1], tf.shape(primary_caps)[2], self.num_classes),
            dtype=tf.float32
        )
        relevance = tf.expand_dims(tf.expand_dims(relevance_scores, axis=-1), axis=-1)

        for i in range(self.routing_iters):
            c = tf.nn.softmax(b, axis=-1)
            c = c * relevance
            c = c / (tf.reduce_sum(c, axis=-1, keepdims=True) + 1e-8)

            s = tf.reduce_sum(tf.expand_dims(c, axis=-1) * u_hat, axis=[1, 2])
            v = squash(s)

            if i < self.routing_iters - 1:
                agreement = tf.reduce_sum(
                    u_hat * tf.expand_dims(tf.expand_dims(v, axis=1), axis=1),
                    axis=-1
                )
                b = b + agreement

        return v, c

In [12]:
class MobileNetV2Hybrid(Model):
    def __init__(self,
                 num_classes=NUM_CLASSES,
                 embed_dim=EMBED_DIM,
                 num_heads=NUM_HEADS,
                 transformer_depth=TRANSFORMER_DEPTH,
                 mlp_dim=MLP_DIM,
                 primary_caps=PRIMARY_CAPS,
                 primary_caps_dim=PRIMARY_CAPS_DIM,
                 class_caps_dim=CLASS_CAPS_DIM,
                 routing_iters=ROUTING_ITERS,
                 **kwargs):
        super().__init__(**kwargs)

        self.num_classes = num_classes

        self.backbone = MobileNetV2(
            weights='imagenet',
            include_top=False,
            input_shape=(224, 224, 3)
        )
        self.backbone.trainable = False

        self.proj_conv = layers.Conv2D(96, 1, padding='same', use_bias=False,kernel_regularizer=REG)
        self.proj_bn = layers.BatchNormalization()
        self.proj_act = layers.Activation('swish')
        self.proj_dropout = layers.Dropout(0.25)
        self.tokenizer = PatchTokenization(patch_size=2, embed_dim=embed_dim)

        self.transformer_blocks = [
            TransformerBlock(embed_dim, num_heads, mlp_dim, dropout=DROPOUT, name=f'transformer_block_{i}')
            for i in range(transformer_depth)
        ]

        self.final_norm = layers.LayerNormalization(epsilon=1e-6)
        self.token_relevance_head = layers.Dense(1, activation='sigmoid',kernel_regularizer=REG)

        self.primary_caps = PrimaryCapsule(primary_caps, primary_caps_dim)
        self.class_caps = RelevanceAwareClassCapsule(num_classes, class_caps_dim, routing_iters)

        self.last_conv_feature_map = None
        self.last_attention_scores = None
        self.last_routing_coeffs = None

    def call(self, inputs, training=False, return_extras=False):
        feature_map = self.backbone(inputs, training=training)
        feature_map = self.proj_act(self.proj_bn(self.proj_conv(feature_map), training=training))
        feature_map = self.proj_dropout(feature_map, training=training)
        self.last_conv_feature_map = feature_map

        tokens = self.tokenizer(feature_map)

        attention_scores_list = []
        for i, blk in enumerate(self.transformer_blocks):
            if i == len(self.transformer_blocks) - 1:
                tokens, attn = blk(tokens, training=training, return_attention=True)
                attention_scores_list.append(attn)
            else:
                tokens = blk(tokens, training=training)

        tokens = self.final_norm(tokens)

        token_relevance = tf.squeeze(self.token_relevance_head(tokens), axis=-1)

        if attention_scores_list:
            last_attn = attention_scores_list[-1]
            mean_attn = tf.reduce_mean(last_attn, axis=1)
            attn_importance = tf.reduce_mean(mean_attn, axis=1)
            fused_relevance = 0.5 * attn_importance + 0.5 * token_relevance
        else:
            fused_relevance = token_relevance

        fused_relevance = fused_relevance / (tf.reduce_sum(fused_relevance, axis=-1, keepdims=True) + 1e-8)

        primary_caps = self.primary_caps(tokens)
        class_caps, routing_coeffs = self.class_caps([primary_caps, fused_relevance])

        caps_lengths = tf.norm(class_caps, axis=-1)
        probs = tf.nn.softmax(caps_lengths, axis=-1)

        self.last_attention_scores = attention_scores_list[-1] if attention_scores_list else None
        self.last_routing_coeffs = routing_coeffs

        if return_extras:
            return {
                'logits': probs,
                'caps_lengths': caps_lengths,
                'class_capsules': class_caps,
                'feature_map': feature_map,
                'tokens': tokens,
                'relevance': fused_relevance,
                'attention_scores': self.last_attention_scores,
                'routing_coeffs': routing_coeffs
            }

        return probs

In [13]:
class HybridTrainer(Model):
    def __init__(self, backbone, alpha_ce=ALPHA_CE, beta_margin=BETA_MARGIN, **kwargs):
        super().__init__(**kwargs)
        self.backbone = backbone
        self.alpha_ce = alpha_ce
        self.beta_margin = beta_margin

        self.ce_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

        self.loss_tracker = tf.keras.metrics.Mean(name='loss')
        self.ce_tracker = tf.keras.metrics.Mean(name='ce_loss')
        self.margin_tracker = tf.keras.metrics.Mean(name='margin_loss')
        self.acc_metric = tf.keras.metrics.CategoricalAccuracy(name='accuracy')
        self.auc_metric = AUC(name='auc', curve='ROC', multi_label=False)

    @property
    def metrics(self):
        return [
            self.loss_tracker, self.ce_tracker, self.margin_tracker,
            self.acc_metric,self.auc_metric
        ]

    def capsule_margin_loss(self, y_true, y_pred):
        present_error = tf.square(tf.maximum(0.0, 0.9 - y_pred))
        absent_error = tf.square(tf.maximum(0.0, y_pred - 0.1))
        loss = y_true * present_error + 0.5 * (1.0 - y_true) * absent_error
        return tf.reduce_mean(tf.reduce_sum(loss, axis=1))

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            y_pred = self.backbone(x, training=True)
            ce_loss = self.ce_fn(y, y_pred)
            margin_loss = self.capsule_margin_loss(y, y_pred)
            total_loss = self.alpha_ce * ce_loss + self.beta_margin * margin_loss
            total_loss += tf.add_n(self.backbone.losses) if self.backbone.losses else 0.0

        grads = tape.gradient(total_loss, self.backbone.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.backbone.trainable_variables))

        self.loss_tracker.update_state(total_loss)
        self.ce_tracker.update_state(ce_loss)
        self.margin_tracker.update_state(margin_loss)
        self.acc_metric.update_state(y, y_pred)
        self.auc_metric.update_state(y, y_pred)

        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, y = data
        y_pred = self.backbone(x, training=False)
        ce_loss = self.ce_fn(y, y_pred)
        margin_loss = self.capsule_margin_loss(y, y_pred)
        total_loss = self.alpha_ce * ce_loss + self.beta_margin * margin_loss
        total_loss += tf.add_n(self.backbone.losses) if self.backbone.losses else 0.0

        self.loss_tracker.update_state(total_loss)
        self.ce_tracker.update_state(ce_loss)
        self.margin_tracker.update_state(margin_loss)
        self.acc_metric.update_state(y, y_pred)

        self.auc_metric.update_state(y, y_pred)

        return {m.name: m.result() for m in self.metrics}

    def call(self, x, training=False):
        return self.backbone(x, training=training)

In [ ]:
os.makedirs('/content/working', exist_ok=True)

In [ ]:

# ============================================================
# TRAINING, EVALUATION, EFFICIENCY, AND ROC/AUC HELPERS
# ============================================================

def build_hybrid_training_model():
    backbone = MobileNetV2Hybrid(
        num_classes=NUM_CLASSES,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        transformer_depth=TRANSFORMER_DEPTH,
        mlp_dim=MLP_DIM,
        primary_caps=PRIMARY_CAPS,
        primary_caps_dim=PRIMARY_CAPS_DIM,
        class_caps_dim=CLASS_CAPS_DIM,
        routing_iters=ROUTING_ITERS,
        name='MobileNetV2_Hybrid'
    )
    trainer = HybridTrainer(backbone, alpha_ce=ALPHA_CE, beta_margin=BETA_MARGIN, name='Hybrid_Trainer')
    return trainer, backbone


def count_model_parameters(model):
    total_params = int(np.sum([tf.keras.backend.count_params(w) for w in model.weights]))
    trainable_params = int(np.sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]))
    non_trainable_params = int(np.sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights]))
    model_size_mb_float32 = (total_params * 4) / (1024 ** 2)
    return total_params, trainable_params, non_trainable_params, model_size_mb_float32


def get_efficiency_batch(test_generator=None, image_size=(224, 224), batch_size=16):
    if test_generator is not None:
        try:
            test_generator.reset()
            batch_x, _ = test_generator[0]
            return batch_x.astype(np.float32)
        except Exception as e:
            print('Could not use test_gen batch. Falling back to random input.')
            print('Reason:', e)
    return np.random.normal(size=(batch_size, image_size[0], image_size[1], 3)).astype(np.float32)


def measure_inference_time(model, sample_batch, warmup_runs=10, timed_runs=50):
    sample_batch = tf.convert_to_tensor(sample_batch, dtype=tf.float32)

    @tf.function
    def infer_step(x):
        return model(x, training=False)

    _ = infer_step(sample_batch)
    for _ in range(warmup_runs):
        _ = infer_step(sample_batch)

    start = time.perf_counter()
    for _ in range(timed_runs):
        preds = infer_step(sample_batch)
        _ = preds.numpy()
    total_time = time.perf_counter() - start

    batch_size_actual = int(sample_batch.shape[0])
    avg_batch_time_sec = total_time / timed_runs
    avg_image_time_ms = (avg_batch_time_sec / batch_size_actual) * 1000
    throughput_images_per_sec = batch_size_actual / avg_batch_time_sec

    return avg_batch_time_sec, avg_image_time_ms, throughput_images_per_sec


def measure_full_test_prediction_time(model, test_generator):
    test_generator.reset()
    n_images = len(test_generator.classes)
    start = time.perf_counter()
    _ = model.predict(test_generator, verbose=0)
    total_time_sec = time.perf_counter() - start
    avg_image_time_ms = (total_time_sec / n_images) * 1000
    throughput_images_per_sec = n_images / total_time_sec
    return total_time_sec, avg_image_time_ms, throughput_images_per_sec


def compute_roc_auc_and_plot(y_true, y_prob, class_names, save_path):
    num_classes = len(class_names)
    y_true_onehot = to_categorical(y_true, num_classes=num_classes)

    fpr, tpr, roc_auc = {}, {}, {}
    for i, class_name in enumerate(class_names):
        fpr[i], tpr[i], _ = roc_curve(y_true_onehot[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    fpr['micro'], tpr['micro'], _ = roc_curve(y_true_onehot.ravel(), y_prob.ravel())
    roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])

    macro_auc = roc_auc_score(y_true_onehot, y_prob, average='macro', multi_class='ovr')
    weighted_auc = roc_auc_score(y_true_onehot, y_prob, average='weighted', multi_class='ovr')

    plt.figure(figsize=(9, 7))
    for i, class_name in enumerate(class_names):
        plt.plot(fpr[i], tpr[i], linewidth=2, label=f'{class_name} AUC = {roc_auc[i]:.4f}')
    plt.plot(fpr['micro'], tpr['micro'], linestyle='--', linewidth=2, label=f'Micro-average AUC = {roc_auc["micro"]:.4f}')
    plt.plot([0, 1], [0, 1], linestyle=':', linewidth=2, label='Chance level')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Multi-class ROC/AUC Curve - MobileNetV2 Hybrid Model')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    auc_rows = []
    for i, class_name in enumerate(class_names):
        auc_rows.append({'Class': class_name, 'AUC': round(float(roc_auc[i]), 6)})
    auc_rows.extend([
        {'Class': 'Micro Average', 'AUC': round(float(roc_auc['micro']), 6)},
        {'Class': 'Macro Average', 'AUC': round(float(macro_auc), 6)},
        {'Class': 'Weighted Average', 'AUC': round(float(weighted_auc), 6)},
    ])
    return pd.DataFrame(auc_rows), float(roc_auc['micro']), float(macro_auc), float(weighted_auc)


def evaluate_model_for_seed(model, test_gen, class_names, seed_dir, seed):
    test_gen.reset()
    y_true = test_gen.classes
    y_prob = model.predict(test_gen, verbose=1)
    y_prob = y_prob[:len(y_true)]
    y_pred = np.argmax(y_prob, axis=1)

    report_text = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    print(report_text)

    with open(os.path.join(seed_dir, f'classification_report_seed_{seed}.txt'), 'w') as f:
        f.write(report_text)

    report_dict = classification_report(y_true, y_pred, target_names=class_names, digits=6, output_dict=True)
    accuracy = accuracy_score(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Hybrid Model Confusion Matrix - Seed {seed}')
    plt.tight_layout()
    cm_path = os.path.join(seed_dir, f'confusion_matrix_seed_{seed}.png')
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.show()

    roc_path = os.path.join(seed_dir, f'roc_auc_curve_seed_{seed}.png')
    auc_df, micro_auc, macro_auc, weighted_auc = compute_roc_auc_and_plot(y_true, y_prob, class_names, roc_path)
    auc_df.to_csv(os.path.join(seed_dir, f'roc_auc_results_seed_{seed}.csv'), index=False)
    display(auc_df)

    return {
        'Seed': seed,
        'Accuracy': float(accuracy),
        'Macro Precision': float(report_dict['macro avg']['precision']),
        'Macro Recall': float(report_dict['macro avg']['recall']),
        'Macro F1': float(report_dict['macro avg']['f1-score']),
        'Weighted Precision': float(report_dict['weighted avg']['precision']),
        'Weighted Recall': float(report_dict['weighted avg']['recall']),
        'Weighted F1': float(report_dict['weighted avg']['f1-score']),
        'Micro AUC': micro_auc,
        'Macro AUC': macro_auc,
        'Weighted AUC': weighted_auc,
        'Support': int(len(y_true)),
        'Confusion Matrix Path': cm_path,
        'ROC Curve Path': roc_path,
    }, y_true, y_pred, y_prob


def get_efficiency_result(model, test_gen, seed):
    sample_batch = get_efficiency_batch(test_generator=test_gen, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE)
    _ = model(sample_batch, training=False)

    total_params, trainable_params, non_trainable_params, model_size_mb = count_model_parameters(model)
    avg_batch_time_sec, avg_image_time_ms, throughput_images_per_sec = measure_inference_time(
        model, sample_batch, warmup_runs=10, timed_runs=50
    )
    full_test_time_sec, full_test_avg_image_ms, full_test_throughput = measure_full_test_prediction_time(model, test_gen)
    device_name = 'GPU' if len(tf.config.list_physical_devices('GPU')) > 0 else 'CPU'

    return {
        'Seed': seed,
        'Model': 'MobileNetV2 + Transformer + Capsule Hybrid',
        'Device': device_name,
        'Total Parameters': total_params,
        'Trainable Parameters': trainable_params,
        'Non-trainable Parameters': non_trainable_params,
        'Approx. Model Size (MB, float32)': round(model_size_mb, 4),
        'Batch Size Used': int(sample_batch.shape[0]),
        'Avg Batch Inference Time (sec)': round(avg_batch_time_sec, 6),
        'Avg Single Image Inference Time (ms)': round(avg_image_time_ms, 4),
        'Throughput (images/sec)': round(throughput_images_per_sec, 4),
        'Full Test Prediction Time incl. generator (sec)': round(full_test_time_sec, 6),
        'Full Test Avg Image Time incl. generator (ms)': round(full_test_avg_image_ms, 4),
        'Full Test Throughput incl. generator (images/sec)': round(full_test_throughput, 4),
    }


In [ ]:

# ============================================================
# RUN THE SAME MODEL FOR 5 SEEDS
# ============================================================

all_seed_results = []
all_efficiency_results = []
trained_seed_models = {}   # Optional: keeps trained models in memory. Clear this if RAM is low.

for seed in SEEDS:
    print('\n' + '=' * 90)
    print(f'SEED {seed}')
    print('=' * 90)

    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    gc.collect()

    seed_dir = f'/content/working/seed_{seed}'
    os.makedirs(seed_dir, exist_ok=True)

    train_gen, val_gen, test_gen = build_generators(
        TRAIN_PATH,
        TEST_PATH,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        seed=seed
    )

    CLASS_NAMES = list(train_gen.class_indices.keys())
    print('Class indices:', train_gen.class_indices)

    hybrid_trainer, hybrid_backbone = build_hybrid_training_model()
    _ = hybrid_trainer(tf.random.normal((2, 224, 224, 3)))

    # -----------------------------
    # Phase 1: Train top/custom layers
    # -----------------------------
    hybrid_trainer.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4)
    )

    phase1_ckpt = os.path.join(seed_dir, f'mobilenetv2_hybrid_seed_{seed}_phase1.weights.h5')
    callbacks1 = [
        EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, mode='min'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1, mode='min'),
        ModelCheckpoint(phase1_ckpt, monitor='val_loss', mode='min', save_best_only=True, save_weights_only=True, verbose=1)
    ]

    history1 = hybrid_trainer.fit(
        train_gen,
        validation_data=val_gen,
        epochs=15,
        callbacks=callbacks1,
        verbose=1
    )

    # -----------------------------
    # Phase 2: Fine-tuning
    # -----------------------------
    hybrid_backbone.backbone.trainable = True
    for layer in hybrid_backbone.backbone.layers[:100]:
        layer.trainable = False

    hybrid_trainer.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=5e-5, weight_decay=1e-4)
    )

    phase2_ckpt = os.path.join(seed_dir, f'mobilenetv2_hybrid_seed_{seed}_phase2.weights.h5')
    callbacks2 = [
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, mode='min'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4, min_lr=1e-7, verbose=1, mode='min'),
        ModelCheckpoint(phase2_ckpt, monitor='val_loss', mode='min', save_best_only=True, save_weights_only=True, verbose=1)
    ]

    history2 = hybrid_trainer.fit(
        train_gen,
        validation_data=val_gen,
        epochs=15,
        callbacks=callbacks2,
        verbose=1
    )

    hybrid_trainer.load_weights(phase2_ckpt)

    # Save training history for this seed
    history_df = pd.concat([
        pd.DataFrame(history1.history).assign(Phase='Phase 1'),
        pd.DataFrame(history2.history).assign(Phase='Phase 2')
    ], ignore_index=True)
    history_df.to_csv(os.path.join(seed_dir, f'training_history_seed_{seed}.csv'), index=False)

    # Evaluation + ROC/AUC
    seed_result, y_true, y_pred, y_prob = evaluate_model_for_seed(
        hybrid_trainer,
        test_gen,
        CLASS_NAMES,
        seed_dir,
        seed
    )
    all_seed_results.append(seed_result)

    # Model efficiency: parameters + inference time
    efficiency_result = get_efficiency_result(hybrid_backbone, test_gen, seed)
    all_efficiency_results.append(efficiency_result)

    # Optional: store final model references for later use inside runtime
    trained_seed_models[seed] = {
        'trainer': hybrid_trainer,
        'backbone': hybrid_backbone,
        'test_gen': test_gen,
        'class_names': CLASS_NAMES,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_prob': y_prob,
    }

    print('\nSeed result:')
    display(pd.DataFrame([seed_result]))
    print('\nEfficiency result:')
    display(pd.DataFrame([efficiency_result]))


In [ ]:

# ============================================================
# 5-SEED SUMMARY: MEAN ± STD
# ============================================================

results_df = pd.DataFrame(all_seed_results)
efficiency_df = pd.DataFrame(all_efficiency_results)

results_csv_path = '/content/working/5_seed_classification_results.csv'
efficiency_csv_path = '/content/working/5_seed_model_efficiency_results.csv'
results_df.to_csv(results_csv_path, index=False)
efficiency_df.to_csv(efficiency_csv_path, index=False)

print('=' * 90)
print('5-SEED CLASSIFICATION RESULTS')
print('=' * 90)
display(results_df)

metric_cols = [
    'Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1',
    'Weighted Precision', 'Weighted Recall', 'Weighted F1',
    'Micro AUC', 'Macro AUC', 'Weighted AUC'
]

summary_rows = []
for metric in metric_cols:
    summary_rows.append({
        'Metric': metric,
        'Mean': results_df[metric].mean(),
        'Std': results_df[metric].std(ddof=1),
        'Mean (%)': results_df[metric].mean() * 100,
        'Std (%)': results_df[metric].std(ddof=1) * 100,
    })

summary_df = pd.DataFrame(summary_rows)
summary_csv_path = '/content/working/5_seed_mean_std_summary.csv'
summary_df.to_csv(summary_csv_path, index=False)

print('\n' + '=' * 90)
print('5-SEED MEAN ± STD SUMMARY')
print('=' * 90)
display(summary_df)

print('\n' + '=' * 90)
print('5-SEED MODEL EFFICIENCY RESULTS')
print('=' * 90)
display(efficiency_df)

print('\nSaved files:')
print('Classification results:', results_csv_path)
print('Efficiency results:', efficiency_csv_path)
print('Mean ± STD summary:', summary_csv_path)

print('\nPaper-ready accuracy format:')
acc_mean = results_df['Accuracy'].mean() * 100
acc_std = results_df['Accuracy'].std(ddof=1) * 100
print(f'Accuracy = {acc_mean:.2f}% ± {acc_std:.2f}%')


## Output files

After running the notebook, Colab will save all reports under `/content/working/`, including per-seed classification reports, confusion matrices, ROC/AUC curves, training histories, efficiency CSVs, and the final 5-seed mean ± std summary.